In [1]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

session = boto3.Session(
    profile_name=os.environ["AWS_PROFILE_NAME"],
    region_name=os.environ["AWS_REGION"],
)

In [3]:
client = session.client("bedrock-runtime")

model_id = os.environ["MODEL_ID"]

def add_message(messages , text , role):
    new_message = {
        'role' : role ,
         'content' : [
             {
                 'text' : text
             }
         ]
    }
    messages.append(new_message)
    
def chat(messages , sys_prompt = "" , temp = 0.1 , stop = []):
    
    params = {
        "modelId": model_id,
        "messages" : messages,
        "inferenceConfig" : {
            "temperature" : temp,
            "stopSequences": stop 
         }
        
    }
     
    if sys_prompt :
        params["system"] = [{"text" : sys_prompt}]
        
    response = client.converse( **params )
          
    
    return response['output']['message']['content'][0]['text']

def chat_stream(messages , sys_prompt = "" , temp = 0.1):
    
    params = {
        "modelId": model_id,
        "messages" : messages,
        "inferenceConfig" : {
            "temperature" : temp
         }
        
    }
     
    if sys_prompt :
        params["system"] = [{"text" : sys_prompt}]
        
    response = client.converse_stream( **params )
    
    
    answer = []
    
    for event in response["stream"] : 
                if "contentBlockDelta" in event :
                    chunk = event["contentBlockDelta"]["delta"]["text"]
                    print(chunk , end="")
                    answer.append(chunk)
                    
    full_chunk = "".join(answer)
   
    
    return full_chunk



def conversation(messages , sys_prompt = "" , temp = 0.1):
    print("Assistant >> Welcome to AWS help Desk! ")
    while True:
        user_input = input("You >>").strip()
        
        print("You >> " , user_input , "\n" )
        
        if user_input.lower() in ['exit' , 'quit' , 'q' , 'e' ] :
            print("Assistant >> Bye!")
            break
        
            
        add_message( messages , user_input , 'user')
        
        
        assistant_reponse= chat( messages , sys_prompt , temp)
        
        add_message(messages , assistant_reponse , 'assistant')
        
        print(f'Assistant >> {assistant_reponse}\n')


In [ ]:

import json 

def generate_dataset():
    prompt = """
    Generate 3 AWS-related tasks that require Python, JSON, or Regex solutions.
    
    Focus on tasks that can be solved by writing a single Python function, 
    a single JSON object, or tasks that do not require writing much code.
    
    Example output:
    [
        {
            "task": "Description of task"
            "format" : JSON or Python or Regex 
    
        },
        
    ]
    
    Please generate 3 objects.
    """
    messages = []
    add_message(messages , prompt , 'user')
    add_message(messages , "```json" , 'assistant')
    output = chat(messages , stop=["```"])
    return json.loads(output)
    




# prompt eval workflow 

# prompt draft

# eval set ( possible inputs that we might want to put into our prompt)

# feed through claude to get actual answer

# grade them ( using grader ) -- gives number base on quality of answer

# make average of the scores (to say how well)

# iterate for number to increase


# eval workflow for use cases in aws --> output should be : Python , JSON or a regular expression with no explanation

# now we have to assemble a dataset 




In [6]:
dataset = generate_dataset()

with open("dataset.json" , "w") as f:
    json.dump(dataset,f , indent=2)

In [20]:
def run_prompt(test_case):
    prompt = f"""
    Provide a solution to the following task:
      {test_case["task"]}
    In the following format :
      {test_case["format"]}
      
    * only provide answer in python or json or regex
    * do not add additional comments or commentry 
    """
    messages = []
    add_message(messages , prompt , "user")
    add_message(messages, "```code" , "assistant")
    response =  chat(messages , stop=["```"])
    return response
    

In [19]:
def grade_by_model(test_case, output):
    eval_prompt = f""" 
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    messages = []
    add_message(messages , eval_prompt , "user")
    add_message(messages ,"```json" , "assistant" )
    
    eval_res = chat(messages , stop=["```"])
    return json.loads(eval_res)
    

In [30]:
import ast
import re


def validate_json(text) : 
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_by_code(test_case , output ):
    res = 0
    if test_case["format"] == "JSON":
        return validate_json(output)
    elif test_case["format"] == "Python":
        return validate_python(output)
    
    return validate_regex(output)
    
    
    



In [ ]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case , output)
    print(model_grade)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    syntax_score = grade_by_code(test_case , output)
    
    avg_score = (model_score + syntax_score) / 2 
    
    
    return {
        "output": output,
        "test_case" : test_case["task"],
        "score" : avg_score,
        "reasoning" : reasoning
    }
    
    

In [27]:

from statistics import mean

def run_eval(dataset):
    
    eval_results = []
    
    for value in dataset:
        res = run_test_case(value)
        eval_results.append(res)
    
    mean_score = mean([result["score"] for result in eval_results])
    
    print(f"Average Score : {mean_score}")
    
    return eval_results
        
        
        

In [31]:
with open("dataset.json" , "r") as file :
        dataset = json.load(file)
        
results = run_eval(dataset)

Average Score : 7.75


In [32]:
print(json.dumps(results , indent=2))

[
  {
    "output": "\nimport re\n\nlog_entry = '2024-01-15T10:30:45Z ERROR Database connection failed: timeout after 30s'\n\npattern = r'(\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}Z)\\s+(\\w+)\\s+(.+)'\n\nmatch = re.match(pattern, log_entry)\n\nif match:\n    timestamp, log_level, error_message = match.groups()\n    result = {\n        \"timestamp\": timestamp,\n        \"log_level\": log_level,\n        \"error_message\": error_message\n    }\n    print(result)\n",
    "test_case": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and error message using regex. Example log: '2024-01-15T10:30:45Z ERROR Database connection failed: timeout after 30s'",
    "score": 8.25,
    "reasoning": "The solution demonstrates a solid understanding of regex fundamentals and handles the happy path well. However, it lacks robustness for production use. The overly permissive log_level pattern and missing end-of-string anchor mean invalid logs could be partially matched. Adding inp